In [ ]:
import torch
import math
import random
from torch import nn
from tqdm import tqdm
from torchvision import transforms
import torchvision.transforms as T
from torchvision.datasets import MNIST
from torchvision.utils import make_grid
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from PIL import Image 
import PIL
import numpy as np
import torchvision

In [ ]:
torch.__version__

In [ ]:
torchvision.__version__

In [ ]:
torch.cuda.is_available()

# Details of the HyperParameters:
<ul>
    <li>Epochs controls the number of iterations for the training loop</li>
    <li>Display_step determines after how many steps the images will be displayed for comparison</li>
    <li>Batch_size controls images per batch for training</li>
    <li>Crit_repeats determines how many times critic is updated per backward pass of generator. This is an element of Wasserstein Loss</li>
    <li>Learning_rate determines how fast the model learns and approaches convergence. Optimal learning_rate for ACGAN is 0.0002</li>
    <li>Beta_1 and Beta_2 are the hyperparameters for Adam Optimizer. Optimal values according to ACGAN Paper are already set</li>
    <li>Lambda_ is a hyperparameter for controlling the gradient penalty in Wasserstein Loss</li>
    <li>z_dim is the noise vector's dimension and nDz and nGz are the hidden dimensions for the Critic and Generator respectively</li>
    <li>Image_channels is 1 in our case since we are dealing with gray-scale images</li>
    <li>Image_size controls the size of our image</li>
    <li>Conditional will always be True in our case since we are using ACGAN. I used it initially when building the network but leave it as True</li>
    <li>n_images is the number of images to be generated for checking the performance later on. It must be equal to the batch_size</li>
    <li>Size is the shape or the dimensions of an image</li>
    <li>NumClasses specify how many classes we are going to use. In our case, 15</li>
    <li>Device specifies whether we are training on CPU or GPU. For CPU use 'cpu' and for GPU use 'cuda'</li>
    <li>Loss specifies which loss are we using. I am using Negative Log Likelihood for Labels' comparison and BCELoss or Wasserstein Loss for Generator and Critic. Pass 'BCE' for BCELoss and 'W' for Wasserstein Loss</li>
    <li>Trained determines if we have already trained the model and now we are further training it. In this case, save the model_states using the code in third-last cell and give paths to the model as arguments</li>

In [ ]:
epochs = 20
display_step = 100
batch_size = 16
crit_repeats = 5
learning_rate = 0.0002
beta_1 = 0.9
beta_2 = 0.999
lambda_ = 10
z_dim = 100
img_channels = 1
nGz = 16
nDz = 16
image_size = 112
conditional = True
n_images = batch_size
size = (1, 112, 112)
numClasses = 15
device = 'cuda'
loss = 'W'
trained = True

In [ ]:
DATA_DIR = 'NIH Dataset/Images'
stats = (0.5), (0.5)

In [ ]:
train_ds = ImageFolder(DATA_DIR, transform = T.Compose([
    T.Grayscale(num_output_channels = 1),
    T.Resize(image_size),
    T.CenterCrop(image_size),
    T.ToTensor(),
    T.Normalize(*stats)])
)

dataloader = DataLoader(
    train_ds, 
    batch_size, 
    shuffle = True, 
    num_workers = 2, 
    pin_memory = True
)

In [ ]:
for real, labels in tqdm(dataloader):
    break

In [ ]:
real.shape

In [ ]:
train_ds.classes

In [ ]:
def show_tensor_images(image_tensor, num_images = 16, size = (1, 28, 28)):
    image_tensor = (image_tensor + 1) / 2
    image_unflat = image_tensor.detach().cpu()
    image_grid = make_grid(image_unflat[:num_images], nrow = 5)
    images = image_unflat[:num_images]
    plt.imshow(image_grid.permute(1, 2, 0).squeeze())
    plt.show()
    
    #Uncomment if you want to save the images that are generated
#     for i in range(images.shape[0]):
#         image = (images[i] * 0.5) + 0.5
#         image = image.squeeze().numpy()
#         image = (image * 255).astype(np.uint8)
#         image = Image.fromarray(image, mode = 'L')
#         image.save('GeneratedImages/image_' + str(i) + '.png')

In [ ]:
def combineVectors(v1, v2):
    return torch.cat((v1, v2), axis = 1)

In [ ]:
def oneHotEncode(numClasses, labels):
    return nn.functional.one_hot(labels, numClasses)

In [ ]:
def getInputDimensions(z_dim, shape, numClasses):
    generator_input_dim = z_dim + numClasses
    discriminator_input_dim = shape[0] + numClasses
    return generator_input_dim, discriminator_input_dim

In [ ]:
def generate_noise(n_examples, z_dim, device = 'cpu'):
    noise = torch.randn(n_examples, z_dim, device = device)
    return noise

In [ ]:
def weights_init(m):
    if isinstance(m, nn.Conv2d) or isinstance(m, nn.ConvTranspose2d):
        torch.nn.init.normal_(m.weight, 0.0, 0.02)
    if isinstance(m, nn.BatchNorm2d):
        torch.nn.init.normal_(m.weight, 0.0, 0.02)
        torch.nn.init.constant_(m.bias, 0)

In [ ]:
# class Generator(nn.Module):
#     def __init__(self, z_dim = 10, img_channels = 1, hidden_dims = 64):
#         super(Generator, self).__init__()
#         self.z_dim = z_dim
#         self.gen = nn.Sequential (
#             self.make_gen_block(z_dim, hidden_dims * 4),
#             self.make_gen_block(hidden_dims * 4, hidden_dims * 8, kernel_size = 3, stride = 2),
#             self.make_gen_block(hidden_dims * 8, hidden_dims * 16, kernel_size = 4, stride = 2, padding = 1),
#             self.make_gen_block(hidden_dims * 16, hidden_dims * 32, kernel_size = 4, stride = 2, padding = 1),
#             self.make_gen_block(hidden_dims * 32, hidden_dims, kernel_size = 4, stride = 2, padding = 1),
#             self.make_gen_block(hidden_dims, img_channels, kernel_size = 4, stride = 2, padding = 1, final_layer = True)
#         )
#     def make_gen_block(self, input_channels, output_channels, kernel_size = 3, stride = 2, padding = 0, final_layer = False):
#         if not final_layer:
#             return nn.Sequential(
#                 nn.ConvTranspose2d(input_channels, output_channels, kernel_size, stride, padding),
#                 nn.BatchNorm2d(output_channels),
#                 nn.ReLU()
#             )
#         else:
#             return nn.Sequential(
#                 nn.ConvTranspose2d(input_channels, output_channels, kernel_size, stride, padding),
#                 nn.Tanh()
#             )
#     def unsqueeze_noise(self, noise):
#         return noise.view(len(noise), self.z_dim, 1, 1)
#     def forward(self, noise):
#         noise_in = self.unsqueeze_noise(noise)
#         return self.gen(noise_in)

In [ ]:
class Generator(nn.Module):
    def __init__(self, img_size, z_dim, n_channels):
        super(Generator, self).__init__()

        self.init_size = img_size // 4
        self.l1 = nn.Sequential(nn.Linear(z_dim, 128 * self.init_size ** 2))

        self.conv_blocks = nn.Sequential(
            nn.BatchNorm2d(128),
            nn.Upsample(scale_factor = 2),
            nn.Conv2d(128, 128, 3, stride = 1, padding = 1),
            nn.BatchNorm2d(128, 0.8),
            nn.LeakyReLU(0.2, inplace = True),
            nn.Upsample(scale_factor = 2),
            nn.Conv2d(128, 64, 3, stride = 1, padding = 1),
            nn.BatchNorm2d(64, 0.8),
            nn.LeakyReLU(0.2, inplace = True),
            nn.Conv2d(64, n_channels, 3, stride = 1, padding = 1),
            nn.Tanh(),
        )

    def forward(self, z):
        out = self.l1(z)
        out = out.view(out.shape[0], 128, self.init_size, self.init_size)
        img = self.conv_blocks(out)
        return img

In [ ]:
class Critic(nn.Module):
    def __init__(self, img_channels = 1, hidden_dims = 64):
        super(Critic, self).__init__()
        self.crit = nn.Sequential (
            self.make_crit_block(img_channels, hidden_dims),
            self.make_crit_block(hidden_dims, hidden_dims * 2),
            self.make_crit_block(hidden_dims * 2, hidden_dims * 4),
            self.make_crit_block(hidden_dims * 4, hidden_dims * 8),
        )
        self.fc = nn.Linear(hidden_dims * 8 * 7 * 7, 1)
        self.fc_aux = nn.Linear(hidden_dims * 8 * 7 * 7, 15)
        self.sigmoid_1 = nn.Sigmoid()
    def make_crit_block(self, input_channels, output_channels, kernel_size = 4, stride = 2, padding = 1, final_layer = False):
        if not final_layer:
            return nn.Sequential(
                nn.Conv2d(input_channels, output_channels, kernel_size, stride, padding),
                nn.BatchNorm2d(output_channels),
                nn.LeakyReLU(0.2, inplace = True),
                nn.Dropout(0.5, inplace = False)
            )
        else:
            return nn.Sequential(
                nn.Conv2d(input_channels, output_channels, kernel_size, stride, padding)
            )
    def forward(self, img):
        img_ = self.crit(img)
        img_fc = self.fc(img_.reshape((img_.shape[0], img_.shape[1] * img_.shape[2] * img_.shape[3])))
        label_imgs = self.fc_aux(img_.reshape((img_.shape[0], img_.shape[1] * img_.shape[2] * img_.shape[3])))
        real_fake = self.sigmoid_1(img_fc).view(-1, 1).squeeze(1)
        return real_fake, label_imgs

In [ ]:
def get_gradient(crit, real, fake, epsilon):
    interpolated_img = real * epsilon + fake * (1 - epsilon)
    pred, _ = crit(interpolated_img)
    grad = torch.autograd.grad(
        inputs = interpolated_img,
        outputs = pred,
        grad_outputs = torch.ones_like(pred), 
        create_graph = True,
        retain_graph = True        
    )[0]
    return grad

In [ ]:
def gradient_penalty(gradient):
    gradient = gradient.view(len(gradient), -1)
    gradient_norm = gradient.norm(2, dim=1)   
    penalty = torch.mean((gradient_norm - 1) ** 2)
    return penalty

In [ ]:
def wasserstein_loss_gen(fake_pred):
    return -torch.mean(fake_pred)

In [ ]:
def wasserstein_loss_crit(fake_pred, real_pred, penalty, lambda_ = 0.1):
    crit_loss = torch.mean(fake_pred) - torch.mean(real_pred) + torch.mean(lambda_ * (penalty))
    return crit_loss

In [ ]:
def generate_images(trained, path_gen, nGz, n_examples, z_dim, conditional, numClasses = 0, num_images = 25, size = (1, 28, 28), labels = [], device = 'cpu'):
    if trained:
        img_channels = 1
        generator_input_dim, _ = getInputDimensions(z_dim, size, numClasses)
        generator = Generator(size[1], generator_input_dim, img_channels).to(device)
        generator.load_state_dict(torch.load(path_gen, map_location = torch.device(device)))
    if conditional == True:
        if len(labels) == 0:
            for i in range(num_images):
                labels.append(random.randint(0, numClasses - 1))
            labels = torch.Tensor(labels).to(device)
        elif len(labels) == 1:
            labels = torch.floor(labels[0] + torch.rand(num_images, ))
            labels = labels.to(device)
        labels = labels.to(torch.int64)
        one_hot = oneHotEncode(numClasses, labels)
        pred_noise = generate_noise(n_examples, z_dim, device)
        pred_noise_labels = combineVectors(pred_noise, one_hot)
        pred_images = generator(pred_noise_labels)
        return pred_images
    else:
        pred_noise = generate_noise(n_examples, z_dim, device)
        pred_images = generator(pred_noise)
    show_tensor_images(pred_images, num_images, size)

In [ ]:
def GAN(trained, epochs, nDz, nGz, display_step, img_channels, crit_repeats, learning_rate, beta_1, beta_2, lambda_, z_dim, shape, numClasses, dataloader, loss, device):
    generator_input_dim, _ = getInputDimensions(z_dim, shape, numClasses)
    gen = Generator(shape[1], generator_input_dim, img_channels).to(device)
    crit = Critic(img_channels, nDz).to(device)

    gen_optimizer = torch.optim.Adam(gen.parameters(), lr = learning_rate, betas = (beta_1, beta_2))
    crit_optimizer = torch.optim.Adam(crit.parameters(), lr = learning_rate, betas = (beta_1, beta_2))
    
    if not trained:
        gen = gen.apply(weights_init)
        crit = crit.apply(weights_init)
    else:
        gen.load_state_dict(torch.load('./Generator_112_' + str(loss) + '_.pth'))
        crit.load_state_dict(torch.load('./Critic_112_' + str(loss) + '_.pth'))
        gen_optimizer.load_state_dict(torch.load('./Generator_Optimizer_112_' + str(loss) + '_.pth'))
        crit_optimizer.load_state_dict(torch.load('./Critic_Optimizer_112_' + str(loss) + '_.pth'))
        gen.train()
        crit.train()
    
    if loss == 'BCE':
        criterion = nn.BCELoss()
    criterion_aux = nn.NLLLoss()
    
    cur_step = 0
    generator_losses = []
    critic_losses = []
    for epoch in range(epochs):
        for real, labels in tqdm(dataloader):
            cur_batch_size = len(real)
            real = real.to(device)
            one_hot = oneHotEncode(numClasses, labels.to(device)) 
            mean_iteration_critic_loss = 0
            if loss == 'W':
                for _ in range(crit_repeats):
                    crit_optimizer.zero_grad()
                    fake_noise = generate_noise(cur_batch_size, z_dim, device = device)
                    epsilon = torch.rand(len(real), 1, 1, 1, device = device, requires_grad = True)
                    fake_noise_labels = combineVectors(fake_noise, one_hot)
                    fake_imgs = gen(fake_noise_labels)
                    fake_pred, label_imgs_fake = crit(fake_imgs)
                    real_pred, label_imgs_real = crit(real)
                    grad = get_gradient(crit, real, fake_imgs, epsilon)
                    penalty = gradient_penalty(grad)
                    crit_loss_wasserstein = wasserstein_loss_crit(fake_pred, real_pred, penalty, lambda_)
                    crit_loss_labels_fake = criterion_aux(label_imgs_fake, labels.to(device))
                    crit_loss_labels_real = criterion_aux(label_imgs_real, labels.to(device))
                    crit_loss = crit_loss_wasserstein + crit_loss_labels_fake + crit_loss_labels_real
                    mean_iteration_critic_loss += crit_loss.item() / crit_repeats
                    critic_losses += [mean_iteration_critic_loss]
                    crit_loss.backward(retain_graph = True)
                    crit_optimizer.step()
                cur_step += 1
                
                gen_optimizer.zero_grad()
                fake_noise_1 = generate_noise(cur_batch_size, z_dim, device = device)
                fake_noise_labels_1 = combineVectors(fake_noise_1, one_hot)
                fake_imgs_1 = gen(fake_noise_labels_1)
                fake_pred_1, labels_imgs_fake_1 = crit(fake_imgs_1)
                gen_loss = wasserstein_loss_gen(fake_pred_1)
                gen_loss_labels = criterion_aux(labels_imgs_fake_1, labels.to(device))
                gen_loss = gen_loss + gen_loss_labels
                gen_loss.backward(retain_graph = True)
                gen_optimizer.step()
                generator_losses += [gen_loss.item()]
                
            elif loss == 'BCE':
                crit_optimizer.zero_grad()
                fake_noise = generate_noise(cur_batch_size, z_dim, device = device)
                fake_noise_labels = combineVectors(fake_noise, one_hot)
                fake_imgs = gen(fake_noise_labels)
                fake_pred, label_imgs_fake = crit(fake_imgs)
                real_pred, label_imgs_real = crit(real)
                crit_loss_fake = criterion(fake_pred, torch.zeros_like(fake_pred))
                crit_loss_real = criterion(real_pred, torch.ones_like(real_pred))
                crit_loss_fake_labels = criterion_aux(label_imgs_fake, labels.to(device))
                crit_loss_real_labels = criterion_aux(label_imgs_real, labels.to(device))
                crit_loss_for_fake = crit_loss_fake + crit_loss_fake_labels
                crit_loss_for_fake.backward(retain_graph = True)
                crit_loss_for_real = crit_loss_real + crit_loss_real_labels
                crit_loss_for_real.backward(retain_graph = True)
                crit_loss = crit_loss_for_fake + crit_loss_for_real
                critic_losses += [crit_loss.item()]
                crit_optimizer.step()
                cur_step += 1
                
                gen_optimizer.zero_grad()
                fake_pred_1, label_imgs_fake_1 = crit(fake_imgs)
                gen_loss_fake = criterion(fake_pred_1, torch.ones_like(fake_pred_1))
                gen_loss_labels = criterion_aux(label_imgs_fake_1, labels.to(device))
                gen_loss = gen_loss_fake + gen_loss_labels
                gen_loss.backward(retain_graph = True)
                gen_optimizer.step()
                generator_losses += [gen_loss.item()]
            
            if cur_step % display_step == 0 and cur_step > 0:
                gen_mean = sum(generator_losses[-display_step:]) / display_step
                crit_mean = sum(critic_losses[-display_step:]) / display_step
                print(f"Epoch {epoch}, step {cur_step}: Generator loss: {gen_mean}, critic loss: {crit_mean}")
                show_tensor_images(fake_imgs, size = shape)
                show_tensor_images(real, size = shape)
                step_bins = 20
                num_examples = (len(generator_losses) // step_bins) * step_bins
                plt.plot(
                    range(num_examples // step_bins), 
                    torch.Tensor(generator_losses[:num_examples]).view(-1, step_bins).mean(1),
                    label="Generator Loss"
                )
                plt.plot(
                    range(num_examples // step_bins), 
                    torch.Tensor(critic_losses[:num_examples]).view(-1, step_bins).mean(1),
                    label="Critic Loss"
                )
                plt.legend()
                plt.show()
        torch.save(gen.state_dict(), './Generator_128_' + str(loss) + '_.pth')
        torch.save(crit.state_dict(), './Critic_128_' + str(loss) + '_.pth')
        torch.save(gen_optimizer.state_dict(), './Generator_Optimizer_128_' + str(loss) + '_.pth')
        torch.save(crit_optimizer.state_dict(), './Critic_Optimizer_128_' + str(loss) + '_.pth')
            
    return gen, crit, gen_optimizer, crit_optimizer

In [ ]:
Generator, Critic, Generator_Optimizer, Critic_Optimizer = GAN(trained, epochs, nDz, nGz, display_step, img_channels, crit_repeats, learning_rate, beta_1, beta_2, lambda_, z_dim, size, numClasses, dataloader, loss, device)

<p>The labels argument of generate_images function specifies which class's images to generate. Pass [0] for the first class, [1] for the second class, [2] for the third class and so on.</p>

In [ ]:
generate_images(True, './Generator_112_W_.pth', nGz, 5, z_dim, conditional, numClasses, 5, size, [12], device = device) #112x112 Images

In [ ]:
import os
generated = [1000, 200, 300, 100, 1200, 150, 100, 390, 3000, 500, 9610, 500, 200, 50, 400]
folders = os.listdir('NIH Dataset/Images_ACGAN/train')
for idx, folder in enumerate(folders):
    for g in range(generated[idx]):
        images = generate_images(True, 'Weights/Generator_112_W_ACGAN.pth', nGz, 1, z_dim, True, 15, 1, size = size, labels = [idx], device = device)
        image_tensor = (images + 1) / 2
        image_unflat = image_tensor.detach().cpu()
        inv_normalize = transforms.Normalize(mean = [-0.5 * 0.5], std = [1 / 0.5])
        image = inv_normalize(image_unflat[0])
        image = image.permute(1, 2, 0).squeeze()
        image = Image.fromarray((image.numpy() * 255).astype(np.uint8), mode = 'L')
        image.save('NIH Dataset/Images_ACGAN/train/' + folder + '/image_' + str(g) + '.png')